# 🧬 PhyloMethod: 系統樹推定手法の網羅的ベンチマーク解析ノートブック
### Pairwise Sequence Alignment + Neighbor-Joining (PSA+NJ) vs Multiple Sequence Alignment + Maximum Likelihood (MSA+ML) / MSA+NJ / FastME

本ノートブックは、大規模進化シミュレーションにより得られた系統樹推定ベンチマーク結果（実験1〜実験10）を統合・統計分析・可視化し、さらに全実験・全条件における example シミュレーション（True Tree, True MSA, 推定 Tree 等）および配列類似度（Sequence Similarity/Identity）を検証するための網羅的解析環境です。

---

### 📊 実験設計と検証項目一覧

| 実験 | 名称 | 主要パラメータ | 置換モデル / 距離モデル | 検証パイプライン | 目的 |
|---|---|---|---|---|---|
| **実験1** | 基本パラメータ空間 | $D \in [0.1..3.0]$, $L \in [100..1500]$, $N=32$ | `LG+G4` ($\alpha=1.0$), Poisson | `PSA+NJ`, `MSA+NJ`, `MSA+ML`, `TRUE_DIST+NJ`, `TRUE_MSA+NJ`, `TRUE_MSA+ML` | 距離×配列長空間における Regime Map と各手法のトポロジー精度解明 |
| **実験2** | Taxon数スケーリング | $N \in [8, 16, 64, 128]$, $D \times L$ | `LG+G4`, Poisson | `PSA+NJ`, `MSA+NJ`, `MSA+ML` | Taxon 数増加に対するスケーラビリティと推定精度の推移 |
| **実験3** | サイト間速度不均一性 | $\alpha \in [0.25, 0.5, 1.0, 2.0]$, $D \times L$ | `LG+G4`, Poisson | `PSA+NJ`, `MSA+NJ`, `MSA+ML` | ガンマ形状母数 $\alpha$（不均一性の強さ）に対する各手法の頑健性 |
| **実験4** | 真のPSAアライメント | $D \times L$, $N=32$ | `LG+G4`, Poisson | `TRUE_PSA+NJ` | ペアワイズアライメント誤差を排除した理想的PSAの精度検証 |
| **実験5** | ガンマ補正距離 | $\alpha \in [0.25..2.0]$, $D \times L$ | `LG+G4`, `gamma_poisson` ($\alpha_{\text{dist}}=1.0$) | `PSA+NJ`, `MSA+NJ`, `MSA+ML` | ガンマ補正距離による距離法の精度変化の検証 |
| **実験6** | ICS (短縮・切断配列) | $\mathrm{ics\_prop} \in [0.0..0.2]$, $D \times L$ | `LG+G4_ICS`, Poisson | `PSA+NJ`, `MSA+NJ`, `MSA+ML` | 内部切断・ドメイン欠落（Invariant Category Sites）への耐性 |
| **実験7** | FastME 距離法比較 (NoOption) | $D \times L$, $N=32$ | `LG+G4`, Poisson | `PSA+FastME_NoOption`, `MSA+FastME_NoOption` | 距離法アルゴリズム（FastME vs RapidNJ）の比較 |
| **実験8** | 高進化距離領域 | $D \in [4.0, 5.0, 6.0]$, $L \in [300..1500]$ | `LG+G4`, Poisson | `PSA+NJ`, `MSA+NJ`, `MSA+ML` | 超高変異領域における飽和現象と各手法の限界 |
| **実験9** | 単一均一置換モデル | $D \times L$, $N=32$ | `LG` (均一速度), Poisson | `PSA+NJ`, `MSA+NJ`, `MSA+ML` | サイト間均一環境におけるパイプライン比較 |
| **実験10** | FastME オプション比較 | $D \in [0.1..6.0]$, $L \in [300..1500]$, $N=32$ | `LG+G4`, Poisson / Gamma | `PSA+FastME_SPR`, `MSA+FastME_LG_G` | FastME 最適化オプション（SPR 近傍探索 / ガンマ距離モデル）のトポロジー精度検証 |

---
## 1. 環境セットアップ & ライブラリインポート

In [ ]:
%matplotlib inline
import os
import sys
import glob
import subprocess
import shutil
import warnings
import itertools
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats

import dendropy
from dendropy.calculate import treecompare
import Bio
from Bio import Phylo, SeqIO

# ワーニング非表示
warnings.filterwarnings('ignore')

# 描画スタイル & グローバルフォントサイズ設定
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica', 'Hiragino Sans', 'Yu Gothic']
plt.rcParams.update({
    'font.size': 11,              # 全体の基本フォントサイズ
    'figure.titlesize': 14,       # 全体タイトル (plt.suptitle) のサイズ
    'axes.titlesize': 12,         # サブプロットのタイトル (ax.set_title) のサイズ
    'axes.labelsize': 11,         # 軸ラベル (X軸・Y軸のラベル) のサイズ
    'xtick.labelsize': 10,        # X軸の目盛り数値のサイズ
    'ytick.labelsize': 10,        # Y軸の目盛り数値のサイズ
    'legend.fontsize': 10,        # 凡例のテキストサイズ
    'legend.title_fontsize': 11,  # 凡例のタイトルサイズ
    'figure.dpi': 120,
    'savefig.dpi': 300
})

# micromamba / conda 環境の PATH を設定（mafft, rapidnj, fastme, iqtree が実行できるように設定）
env_bin_paths = [
    "/opt/homebrew/Cellar/micromamba/2.8.1/envs/phylomethod_env/bin",
    os.path.expanduser("~/miniconda3/envs/phylomethod_env/bin"),
    os.path.expanduser("~/miniforge3/envs/phylomethod_env/bin"),
    os.path.expanduser("~/micromamba/envs/phylomethod_env/bin"),
    "/opt/homebrew/bin",
    os.path.expanduser("~/bin")
]
for p in env_bin_paths:
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + ":" + os.environ["PATH"]

# プロジェクトパスの設定
PROJECT_ROOT = Path(os.path.abspath("..")) if os.path.exists("../results") else Path(os.path.abspath("."))
RESULTS_DIR = PROJECT_ROOT / "results"
ANALYSIS_DIR = PROJECT_ROOT / "analysis"
EXAMPLE_DIR = ANALYSIS_DIR / "example"
SIMILARITY_DIR = ANALYSIS_DIR / "similarity"
EXAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SIMILARITY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root   : {PROJECT_ROOT}")
print(f"Results Dir    : {RESULTS_DIR}")
print(f"Analysis Dir   : {ANALYSIS_DIR}")
print(f"Example Dir    : {EXAMPLE_DIR}")
print(f"Similarity Dir : {SIMILARITY_DIR}")

# 外部ツール検出確認
tools = ["mafft", "rapidnj", "fastme", "iqtree", "iqtree2"]
for t in tools:
    path = shutil.which(t)
    status = f"✓ ({path})" if path else "✗ Not found in PATH"
    print(f"  {t:<10}: {status}")

---
## 2. ベンチマーク結果の集約 & 統一スキーマへの統合

各実験ディレクトリから `*_summary.csv` を読み込み、実験ごとに異なっているカラム構成を共通スキーマに正規化して結合します。
- パイプライン名の **`PWA` をすべて `PSA`（Pairwise Sequence Alignment）に統一** します。
- 実験7の FastME については `FastME` $\to$ `FastME_NoOption` に統一します。
- 実験10の FastME オプション結果（`PSA+FastME_SPR`, `MSA+FastME_LG_G`）も統合します。
- 統合したデータフレームを `analysis/all_experiments_summary.csv` として出力します。

In [ ]:
def load_and_unify_benchmark_results(results_dir=RESULTS_DIR):
    """
    resultsディレクトリ配下の全実験結果CSVを読み込み、統一スキーマに変換して統合する。
    """
    true_psa_path = results_dir / "results_true_psa" / "benchmark_true_psa_summary.csv"
    if not true_psa_path.exists():
        true_psa_path = results_dir / "results_true_pwa" / "benchmark_true_pwa_summary.csv"

    experiment_configs = [
        {
            "id": "Exp1_Default",
            "name": "Exp1: D x L Base (LG+G4)",
            "path": results_dir / "results_default" / "benchmark_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            }
        },
        {
            "id": "Exp2_Taxon",
            "name": "Exp2: Taxon Scaling",
            "path": results_dir / "results_taxon" / "benchmark_taxon_summary.csv",
            "defaults": {
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            }
        },
        {
            "id": "Exp3_Alpha",
            "name": "Exp3: Rate Heterogeneity (alpha)",
            "path": results_dir / "results_alpha" / "benchmark_alpha_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            }
        },
        {
            "id": "Exp4_TruePSA",
            "name": "Exp4: True PSA+NJ",
            "path": true_psa_path,
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            }
        },
        {
            "id": "Exp5_Gamma",
            "name": "Exp5: Gamma Distance (gamma_poisson)",
            "path": results_dir / "results_gamma" / "benchmark_gamma_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "ics_prop": 0.0,
                "dist_model": "gamma_poisson",
                "model": "LG+G4",
            }
        },
        {
            "id": "Exp6_ICS",
            "name": "Exp6: Invariant Category Sites (ICS)",
            "path": results_dir / "results_ics" / "benchmark_ics_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "dist_model": "poisson",
                "model": "LG+G4_ICS",
            }
        },
        {
            "id": "Exp7_FastME",
            "name": "Exp7: FastME vs NJ (NoOption)",
            "path": results_dir / "results_fastme_NoOption" / "benchmark_fastme_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            },
            "pipeline_rename": {
                "PWA+FastME": "PSA+FastME_NoOption",
                "PSA+FastME": "PSA+FastME_NoOption",
                "MSA+FastME": "MSA+FastME_NoOption"
            }
        },
        {
            "id": "Exp8_HighDist",
            "name": "Exp8: High Distance (D=4.0-6.0)",
            "path": results_dir / "results_high_dist" / "benchmark_high_dist_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            }
        },
        {
            "id": "Exp9_SimpleLG",
            "name": "Exp9: Homogeneous Model (LG)",
            "path": results_dir / "results_simple" / "benchmark_high_dist_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG",
            }
        },
        {
            "id": "Exp10_FastMEOptions",
            "name": "Exp10: FastME Options (SPR / LG+G)",
            "path": results_dir / "results_fastme_options" / "benchmark_fastme_options_summary.csv",
            "defaults": {
                "num_taxa": 32,
                "alpha": 1.0,
                "ics_prop": 0.0,
                "dist_model": "poisson",
                "model": "LG+G4",
            },
            "pipeline_rename": {
                "PWA+FastME_SPR": "PSA+FastME_SPR",
                "MSA+FastME_LG_G": "MSA+FastME_LG_G"
            }
        }
    ]

    unified_dfs = []
    
    target_columns = [
        "experiment_id",
        "experiment_name",
        "model",
        "dist_model",
        "num_taxa",
        "distance",
        "length",
        "alpha",
        "ics_prop",
        "replicate",
        "pipeline",
        "rf_distance",
        "nrf_distance",
        "best_model_bic",
        "gamma_alpha"
    ]

    for cfg in experiment_configs:
        p = Path(cfg["path"])
        if not p.exists():
            print(f"⚠️ [Skip] {cfg['id']} : File not found ({p})")
            continue
        
        df = pd.read_csv(p)
        df["experiment_id"] = cfg["id"]
        df["experiment_name"] = cfg["name"]

        # デフォルト値補完
        for col, val in cfg.get("defaults", {}).items():
            if col not in df.columns:
                df[col] = val
            else:
                df[col] = df[col].fillna(val)

        # パイプライン名の統一 ("PWA" -> "PSA")
        df["pipeline"] = df["pipeline"].astype(str).str.replace("PWA", "PSA")
        if "pipeline_rename" in cfg:
            df["pipeline"] = df["pipeline"].replace(cfg["pipeline_rename"])
        
        # 共通カラム抽出
        for c in target_columns:
            if c not in df.columns:
                df[c] = np.nan

        df = df[target_columns]
        unified_dfs.append(df)
        print(f"✓ [Loaded] {cfg['id']:<18}: {len(df):>6} rows | Pipelines: {sorted(df['pipeline'].unique())}")

    if not unified_dfs:
        raise RuntimeError("No summary CSVs could be loaded.")

    combined_df = pd.concat(unified_dfs, ignore_index=True)
    
    # 型変換
    combined_df["num_taxa"] = combined_df["num_taxa"].astype(int)
    combined_df["distance"] = combined_df["distance"].astype(float)
    combined_df["length"] = combined_df["length"].astype(int)
    combined_df["alpha"] = combined_df["alpha"].astype(float)
    combined_df["ics_prop"] = combined_df["ics_prop"].astype(float)
    combined_df["replicate"] = combined_df["replicate"].astype(int)
    combined_df["nrf_distance"] = combined_df["nrf_distance"].astype(float)

    # 念のため再度 PWA -> PSA 置換を徹底
    combined_df["pipeline"] = combined_df["pipeline"].str.replace("PWA", "PSA")

    return combined_df

# 全データの統合ロード
df_all = load_and_unify_benchmark_results()

# 統合CSVの保存
out_csv_path = ANALYSIS_DIR / "all_experiments_summary.csv"
df_all.to_csv(out_csv_path, index=False)
print(f"\nAll experiments combined: {len(df_all):,} rows successfully saved to -> {out_csv_path}")

In [ ]:
# 実験別の概要テーブルを表示
summary_overview = df_all.groupby(["experiment_id", "experiment_name", "model", "dist_model"]).agg(
    num_taxa=("num_taxa", lambda s: sorted(s.unique())),
    distances=("distance", lambda s: sorted(s.unique())),
    lengths=("length", lambda s: sorted(s.unique())),
    alphas=("alpha", lambda s: sorted(s.unique())),
    ics_props=("ics_prop", lambda s: sorted(s.unique())),
    pipelines=("pipeline", lambda s: sorted(s.unique())),
    total_replicates=("replicate", "count")
).reset_index()

summary_overview

---
## 3. 共通可視化ユーティリティ関数

ベンチマーク解析で共通して使用するグラフ描画関数を定義します。
- `plot_nrf_heatmap_grid`: パイプライン別の $D \times L$ ヒートマップ
- `plot_nrf_by_distance`: 進化距離 $D$ に対する NRF distance の推移（配列長別）
- `plot_regime_map`: 2手法（PSA+NJ vs MSA+ML 等）の勝敗領域図（Regime Map）

In [ ]:
# パイプライン別カラーパレット定義 (PSA / PWA 双方に対応して堅牢化)
PIPELINE_COLORS = {
    'PSA+NJ': '#1f77b4',              # Blue
    'PWA+NJ': '#1f77b4',              # Blue alias
    'MSA+NJ': '#ff7f0e',              # Orange
    'MSA+ML': '#2ca02c',              # Green
    'TRUE_DIST+NJ': '#d62728',        # Red
    'TRUE_DIST': '#d62728',           # Red
    'TRUE_MSA+NJ': '#9467bd',         # Purple
    'TRUE_MSA+ML': '#8c564b',         # Brown
    'TRUE_PSA+NJ': '#e377c2',         # Pink
    'TRUE_PWA+NJ': '#e377c2',         # Pink alias
    'PSA+FastME_NoOption': '#17becf', # Cyan
    'PWA+FastME_NoOption': '#17becf', # Cyan alias
    'MSA+FastME_NoOption': '#bcbd22', # Olive
    'PSA+FastME_SPR': '#17becf',      # Cyan / Teal (新手法)
    'PWA+FastME_SPR': '#17becf',      # Cyan alias
    'MSA+FastME_LG_G': '#8c564b',     # Brown / Maroon (新手法)
}

def plot_nrf_heatmap_grid(df_subset, title="Normalized RF Distance (NRF) Heatmap Grid", annot=True):
    """
    パイプラインごとに Distance x Length の NRF distance 平均値ヒートマップを描画
    """
    pipelines = sorted(df_subset['pipeline'].unique())
    n_pipes = len(pipelines)
    cols = min(3, n_pipes)
    rows = (n_pipes + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.8, rows * 4.0), squeeze=False)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    
    for idx, pipe in enumerate(pipelines):
        r, c = idx // cols, idx % cols
        ax = axes[r][c]
        sub = df_subset[df_subset['pipeline'] == pipe]
        pivot = sub.pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
        
        sns.heatmap(pivot, ax=ax, annot=annot, fmt='.3f', cmap='YlOrRd', vmin=0.0, vmax=max(0.6, pivot.max().max()),
                    cbar=True, cbar_kws={'label': 'Mean NRF Distance'})
        ax.set_title(f"Pipeline: {pipe}", fontweight='bold')
        ax.set_xlabel("Sequence Length (L)")
        ax.set_ylabel("Evolutionary Distance (D)")
        ax.invert_yaxis()
    
    # 余分なサブプロットを非表示
    for idx in range(n_pipes, rows * cols):
        r, c = idx // cols, idx % cols
        axes[r][c].axis('off')
        
    plt.tight_layout()
    plt.show()

def plot_nrf_by_distance(df_subset, title="NRF Distance vs Evolutionary Distance (D)"):
    """
    横軸: Evolutionary Distance D, 縦軸: NRF Distance, 色: Pipeline, 列: Sequence Length L
    """
    lengths = sorted(df_subset['length'].unique())
    n_lens = len(lengths)
    cols = min(5, n_lens)
    rows = (n_lens + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.8, rows * 3.5), squeeze=False, sharey=True)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    
    for idx, length in enumerate(lengths):
        r, c = idx // cols, idx % cols
        ax = axes[r][c]
        sub = df_subset[df_subset['length'] == length]
        
        sns.lineplot(
            data=sub, x='distance', y='nrf_distance', hue='pipeline',
            palette=PIPELINE_COLORS, marker='o', ax=ax, errorbar=('ci', 95)
        )
        ax.set_title(f"Length L = {length}", fontweight='bold')
        ax.set_xlabel("Evolutionary Distance (D)")
        ax.set_ylabel("NRF Distance" if c == 0 else "")
        ax.set_ylim(-0.02, min(1.0, max(0.6, sub['nrf_distance'].max() + 0.05)))
        if idx > 0 and ax.get_legend():
            ax.get_legend().remove()
            
    if axes[0][0].get_legend():
        axes[0][0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
        
    plt.tight_layout()
    plt.show()

def plot_regime_map(df_subset, pipe_a='PSA+NJ', pipe_b='MSA+ML', title=None):
    """
    D x L グリッドにおける pipe_a と pipe_b の精度差 (NRF_a - NRF_b) を可視化するRegime Map。
    青: pipe_a (PSA+NJ) が優位 (NRFが小さい)
    緑: pipe_b (MSA+ML) が優位
    """
    if title is None:
        title = f"Regime Map: {pipe_a} vs {pipe_b} (Difference in Mean NRF)"
        
    pivot_a = df_subset[df_subset['pipeline'] == pipe_a].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
    pivot_b = df_subset[df_subset['pipeline'] == pipe_b].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
    
    diff = pivot_a - pivot_b  # 負なら a が優位、正なら b が優位
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 差分ヒートマップ
    vlimit = max(abs(diff.min().min()), abs(diff.max().max()), 0.1)
    sns.heatmap(diff, ax=ax1, annot=True, fmt='+.3f', cmap='RdBu_r', vmin=-vlimit, vmax=vlimit,
                cbar_kws={'label': f'Mean NRF Difference ({pipe_a} - {pipe_b})\n(< 0: {pipe_a} better | > 0: {pipe_b} better)'})
    ax1.set_title(f"NRF Difference ({pipe_a} - {pipe_b})", fontweight='bold')
    ax1.set_xlabel("Sequence Length (L)")
    ax1.set_ylabel("Evolutionary Distance (D)")
    ax1.invert_yaxis()
    
    # 勝者マップ
    winner_map = np.where(diff < -0.01, 1, np.where(diff > 0.01, -1, 0))
    winner_df = pd.DataFrame(winner_map, index=diff.index, columns=diff.columns)
    
    cmap_cat = mpl.colors.ListedColormap(["#2ca02c", "#d3d3d3", "#1f77b4"])
    sns.heatmap(winner_df, ax=ax2, cmap=cmap_cat, cbar=False, annot=False, linewidths=1, vmin=-1, vmax=1)
    
    # ラベル注釈
    for i in range(len(diff.index)):
        for j in range(len(diff.columns)):
            val = winner_df.iloc[i, j]
            label = f"{pipe_a}\nWins" if val == 1 else (f"{pipe_b}\nWins" if val == -1 else "Tie")
            ax2.text(j + 0.5, i + 0.5, label, ha='center', va='center', fontweight='bold', color='black')
            
    ax2.set_title(f"Regime Partition: {pipe_a} (Blue) vs {pipe_b} (Green)", fontweight='bold')
    ax2.set_xlabel("Sequence Length (L)")
    ax2.set_ylabel("Evolutionary Distance (D)")
    ax2.invert_yaxis()
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()

---
## 4. 各実験の個別詳細分析

### 4.1 実験1: 基本パラメータ空間 ($D \times L$, $N=32$, 置換モデル: `LG+G4`, $\alpha=1.0$)
- 実行パイプライン: `PSA+NJ`, `MSA+NJ`, `MSA+ML`, `TRUE_DIST+NJ`, `TRUE_MSA+NJ`, `TRUE_MSA+ML`

In [ ]:
df_exp1 = df_all[df_all['experiment_id'] == 'Exp1_Default']

# 1. ヒートマップグリッド
plot_nrf_heatmap_grid(df_exp1, title="Exp1 (Base D x L): NRF Distance Heatmap Grid")

# 2. 距離別推移プロット
plot_nrf_by_distance(df_exp1, title="Exp1 (Base D x L): NRF Distance vs Evolutionary Distance D")

# 3. Regime Map (PSA+NJ vs MSA+ML)
plot_regime_map(df_exp1, pipe_a='PSA+NJ', pipe_b='MSA+ML', title="Exp1: Regime Map (PSA+NJ vs MSA+ML)")

# 4. 距離法同士の比較: PSA+NJ vs MSA+NJ
plot_regime_map(df_exp1, pipe_a='PSA+NJ', pipe_b='MSA+NJ', title="Exp1: Distance Methods Comparison (PSA+NJ vs MSA+NJ)")

# 5. アライメント誤差のインパクト: MSA+NJ vs TRUE_MSA+NJ
plot_regime_map(df_exp1, pipe_a='MSA+NJ', pipe_b='TRUE_MSA+NJ', title="Exp1: Alignment Error Impact (MSA+NJ vs TRUE_MSA+NJ)")

In [ ]:
# Exp1 & Exp8 における ModelFinder 置換モデル選択 & 推定ガンマ母数 alpha の推移
df_full_ml = pd.concat([
    df_all[(df_all['experiment_id'] == 'Exp1_Default') & (df_all['pipeline'].isin(['MSA+ML', 'TRUE_MSA+ML']))],
    df_all[(df_all['experiment_id'] == 'Exp8_HighDist') & (df_all['pipeline'] == 'MSA+ML')]
], ignore_index=True)

# 1. 選択モデル内訳 & 正解モデル選択率
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
top_models = df_full_ml['best_model_bic'].value_counts().head(8).index
df_full_ml['model_grouped'] = df_full_ml['best_model_bic'].apply(lambda x: x if x in top_models else 'Others')

order = list(top_models) + (['Others'] if 'Others' in df_full_ml['model_grouped'].values else [])
sns.countplot(data=df_full_ml, x='model_grouped', hue='pipeline', order=order,
              palette={'MSA+ML': '#2ca02c', 'TRUE_MSA+ML': '#8c564b'}, ax=ax1)
ax1.set_title("Selected Best-Fit Models Breakdown (BIC, D=0.1-6.0)", fontweight='bold')
ax1.set_xlabel("Model Name")
ax1.set_ylabel("Count (Replicates)")
ax1.tick_params(axis='x', rotation=35)
ax1.grid(True, linestyle='--', alpha=0.5)

df_full_ml['is_true_model'] = (df_full_ml['best_model_bic'] == 'LG+G4').astype(float)
pivot_model_acc = df_full_ml[df_full_ml['pipeline'] == 'MSA+ML'].pivot_table(
    index='distance', columns='length', values='is_true_model', aggfunc='mean'
)
sns.heatmap(pivot_model_acc, ax=ax2, annot=True, fmt='.1%', cmap='Greens', vmin=0.0, vmax=1.0,
            cbar_kws={'label': 'Proportion of LG+G4 Selection'})
ax2.axhline(5, color='gray', linestyle='--', linewidth=1.5)
ax2.set_title("True Model (LG+G4) Selection Rate (MSA+ML)", fontweight='bold')
ax2.set_xlabel("Sequence Length (L)")
ax2.set_ylabel("Evolutionary Distance (D)")
ax2.invert_yaxis()
plt.suptitle("ModelFinder Selection Spectrum (True Simulated Model: LG+G4, D = 0.1 to 6.0)", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 2. 推定ガンマ形状母数 alpha の推移 (True alpha = 1.0)
lengths = [300, 500, 1000, 1500]
fig, axes = plt.subplots(1, len(lengths), figsize=(16, 4.2), sharey=True)

for idx, length in enumerate(lengths):
    ax = axes[idx]
    sub = df_full_ml[df_full_ml['length'] == length]
    sns.lineplot(data=sub, x='distance', y='gamma_alpha', hue='pipeline',
                 palette={'MSA+ML': '#2ca02c', 'TRUE_MSA+ML': '#8c564b'}, marker='o', errorbar=('ci', 95), ax=ax)
    ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='True alpha = 1.0' if idx == 0 else "")
    ax.axvline(3.0, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)
    ax.set_title(f"Length L = {length} aa", fontweight='bold')
    ax.set_xlabel("Evolutionary Distance (D)")
    ax.set_ylabel("Estimated Gamma alpha" if idx == 0 else "")
    ax.set_ylim(0.4, 5.5)
    if idx < len(lengths) - 1 and ax.get_legend():
        ax.get_legend().remove()

if axes[-1].get_legend():
    axes[-1].legend(bbox_to_anchor=(1.05, 1.0), loc='upper left', frameon=True, title="Pipeline")

plt.subplots_adjust(wspace=0.08, top=0.85, right=0.85)
plt.suptitle("Estimated Gamma Shape Parameter alpha Spectrum (D = 0.1 to 6.0, True alpha = 1.0)", fontweight='bold', y=0.98)
plt.show()

---
### 4.2 実験2: Taxon数スケーリング実験 ($N \in [8, 16, 64, 128]$, $D \times L$)

In [ ]:
df_exp2 = df_all[df_all['experiment_id'] == 'Exp2_Taxon']

# Taxon数ごとの NRF Distance の推移
plt.figure(figsize=(14, 4.5))
for idx, L in enumerate([100, 500, 1000]):
    plt.subplot(1, 3, idx + 1)
    sub = df_exp2[(df_exp2['length'] == L) & (df_exp2['distance'] == 1.0)]
    sns.lineplot(data=sub, x='num_taxa', y='nrf_distance', hue='pipeline',
                 palette=PIPELINE_COLORS, marker='o')
    plt.title(f"Distance D=1.0, Length L={L}", fontweight='bold')
    plt.xlabel("Number of Taxa (N)")
    plt.ylabel("NRF Distance" if idx == 0 else "")
    plt.xscale('log', base=2)
    plt.xticks([8, 16, 32, 64, 128], [8, 16, 32, 64, 128])
    if idx > 0 and plt.gca().get_legend():
        plt.gca().get_legend().remove()

plt.suptitle("Exp2: Taxon Count Scaling (N = 8 -> 128 at D=1.0)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
### 4.3 実験3: ガンマ形状母数 $\alpha$ / サイト間速度不均一性実験 ($\alpha \in [0.25, 0.5, 1.0, 2.0]$)

In [ ]:
df_exp3 = df_all[df_all['experiment_id'] == 'Exp3_Alpha']

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), sharey=True)
fig.suptitle("Exp3: Effect of Gamma Shape Parameter alpha (Site-Rate Heterogeneity)", fontsize=14, fontweight='bold')

for idx, alpha_val in enumerate([0.25, 0.5, 1.0, 2.0]):
    r, c = idx // 2, idx % 2
    ax = axes[r][c]
    sub = df_exp3[(df_exp3['alpha'] == alpha_val) & (df_exp3['length'] == 500)]
    sns.lineplot(data=sub, x='distance', y='nrf_distance', hue='pipeline',
                 palette=PIPELINE_COLORS, marker='o', ax=ax)
    ax.set_title(f"Gamma alpha = {alpha_val} (L=500)", fontweight='bold')
    ax.set_xlabel("Evolutionary Distance (D)")
    ax.set_ylabel("NRF Distance" if c == 0 else "")
    if idx > 0 and ax.get_legend():
        ax.get_legend().remove()

axes[0][0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
### 4.4 実験4: 真のペアワイズアライメント (TRUE_PSA+NJ 実験)

In [ ]:
df_exp4 = df_all[df_all['experiment_id'] == 'Exp4_TruePSA']

if len(df_exp4) > 0:
    plot_nrf_heatmap_grid(df_exp4, title="Exp4: True PSA+NJ Benchmark Grid")
    plot_nrf_by_distance(df_exp4, title="Exp4: True PSA+NJ vs Distance")

---
### 4.5 実験5: ガンマ補正距離 (Gamma Distance Benchmark: `gamma_poisson`)

In [ ]:
df_exp5 = df_all[df_all['experiment_id'] == 'Exp5_Gamma']

plot_nrf_heatmap_grid(df_exp5, title="Exp5 (Gamma Distance: gamma_poisson): NRF Distance Heatmap Grid")

---
### 4.6 実験6: ICS条件ベンチマーク (Invariant Category Sites, 内部切断・短縮配列)

In [ ]:
df_exp6 = df_all[df_all['experiment_id'] == 'Exp6_ICS']

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.suptitle("Exp6: Impact of Invariant Category Sites (ICS Proportion) on Tree Accuracy (L=500)", fontsize=14, fontweight='bold', y=1.02)

for idx, ics in enumerate([0.0, 0.05, 0.1, 0.2]):
    ax = axes[idx]
    sub = df_exp6[(df_exp6['ics_prop'] == ics) & (df_exp6['length'] == 500)]
    sns.lineplot(data=sub, x='distance', y='nrf_distance', hue='pipeline',
                 palette=PIPELINE_COLORS, marker='o', ax=ax)
    ax.set_title(f"ICS Proportion = {ics*100:.0f}%", fontweight='bold')
    ax.set_xlabel("Evolutionary Distance (D)")
    ax.set_ylabel("NRF Distance" if idx == 0 else "")
    if idx > 0 and ax.get_legend():
        ax.get_legend().remove()

if axes[0].get_legend():
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
### 4.7 実験7: FastME 距離法ベンチマーク (`FastME_NoOption` vs `RapidNJ`)

In [ ]:
df_exp7 = df_all[df_all['experiment_id'] == 'Exp7_FastME']

plot_nrf_heatmap_grid(df_exp7, title="Exp7: FastME_NoOption Benchmark Heatmap Grid")

---
### 4.8 実験8: 高進化距離領域ベンチマーク ($D \in [4.0, 5.0, 6.0]$)

In [ ]:
df_exp8 = df_all[df_all['experiment_id'] == 'Exp8_HighDist']

plot_nrf_heatmap_grid(df_exp8, title="Exp8: High Evolutionary Distance (D=4.0-6.0) Grid")

# 全距離域スペクトラム比較 (D = 0.1 to 6.0)
target_pipelines_exp1 = ['PSA+NJ', 'MSA+NJ', 'MSA+ML', 'TRUE_MSA+NJ', 'TRUE_MSA+ML']

df_full_spectrum = pd.concat([
    df_all[(df_all['experiment_id'] == 'Exp1_Default') & (df_all['pipeline'].isin(target_pipelines_exp1))],
    df_all[(df_all['experiment_id'] == 'Exp4_TruePSA') & (df_all['pipeline'] == 'TRUE_PSA+NJ')],
    df_all[(df_all['experiment_id'] == 'Exp8_HighDist') & (df_all['pipeline'].isin(['PSA+NJ', 'MSA+NJ', 'MSA+ML']))]
], ignore_index=True)

lengths = [300, 500, 1000, 1500]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True)

for idx, length in enumerate(lengths):
    ax = axes[idx]
    sub_df = df_full_spectrum[df_full_spectrum['length'] == length]
    
    sns.lineplot(
        data=sub_df, x='distance', y='nrf_distance', hue='pipeline', style='pipeline',
        palette=PIPELINE_COLORS, markers=True, dashes=False, ax=ax
    )
    ax.set_title(f"Length L = {length} aa", fontweight='bold')
    ax.set_xlabel("Evolutionary Distance (D)")
    ax.set_ylabel("NRF Distance" if idx == 0 else "")
    ax.set_ylim(-0.02, 1.02)
    
    if idx < len(lengths) - 1 and ax.get_legend():
        ax.get_legend().remove()

if axes[-1].get_legend():
    axes[-1].legend(bbox_to_anchor=(1.05, 1.0), loc='upper left', frameon=True, title="Pipeline")

plt.subplots_adjust(wspace=0.08, top=0.85, right=0.85)
plt.suptitle("Topology Accuracy Spectrum: Estimated vs True MSA / True PSA (D = 0.1 to 6.0)", 
             fontsize=13, fontweight='bold', y=0.98)
plt.show()

---
### 4.9 実験9: 単一均一置換モデル (`LG`)

In [ ]:
df_exp9 = df_all[df_all['experiment_id'] == 'Exp9_SimpleLG']

plot_nrf_heatmap_grid(df_exp9, title="Exp9: Homogeneous Model (LG) Heatmap Grid")

---
### 4.10 実験10: FastME 最適化オプション比較 (`PSA+FastME_SPR`, `MSA+FastME_LG_G` vs `PSA+NJ`, `MSA+NJ`, `MSA+ML`)

FastME の最適化オプション（SPR 近傍探索トポロジー改良 `PSA+FastME_SPR`、および LG+Gamma 距離モデル `MSA+FastME_LG_G`）のトポロジー推定精度を、既存の RapidNJ（`PSA+NJ`, `MSA+NJ`）および最尤法（`MSA+ML`）と全距離域（$D = 0.1 \sim 6.0$）において網羅的に比較します。

In [ ]:
df_exp10 = df_all[df_all['experiment_id'] == 'Exp10_FastMEOptions']

# 1. 実験10単体のヒートマップグリッド
plot_nrf_heatmap_grid(df_exp10, title="Exp10: FastME Options (SPR / LG+G) Heatmap Grid")

# 2. 全距離域スペクトラム比較: FastME 2手法 vs 既存手法 (PSA+NJ, MSA+NJ, MSA+ML)
df_comp_fastme = pd.concat([
    df_all[(df_all['experiment_id'] == 'Exp1_Default') & (df_all['pipeline'].isin(['PSA+NJ', 'MSA+NJ', 'MSA+ML']))],
    df_all[(df_all['experiment_id'] == 'Exp8_HighDist') & (df_all['pipeline'].isin(['PSA+NJ', 'MSA+NJ', 'MSA+ML']))],
    df_all[(df_all['experiment_id'] == 'Exp10_FastMEOptions') & (df_all['pipeline'].isin(['PSA+FastME_SPR', 'MSA+FastME_LG_G']))]
], ignore_index=True)

lengths = [300, 500, 1000, 1500]
fig, axes = plt.subplots(1, len(lengths), figsize=(16, 4.2), sharey=True)

for idx, length in enumerate(lengths):
    ax = axes[idx]
    sub = df_comp_fastme[df_comp_fastme['length'] == length]

    sns.lineplot(
        data=sub, x='distance', y='nrf_distance', hue='pipeline', style='pipeline',
        palette=PIPELINE_COLORS, markers=True, dashes=False, ax=ax, errorbar=('ci', 95), linewidth=1.8
    )
    ax.set_title(f"Length L = {length} aa", fontweight='bold')
    ax.set_xlabel("Evolutionary Distance (D)")
    ax.set_ylabel("NRF Distance" if idx == 0 else "")
    ax.set_ylim(-0.02, 1.02)

    if idx < len(lengths) - 1 and ax.get_legend():
        ax.get_legend().remove()

if axes[-1].get_legend():
    axes[-1].legend(bbox_to_anchor=(1.05, 1.0), loc='upper left', frameon=True, title="Pipeline")

plt.subplots_adjust(wspace=0.08, top=0.85, right=0.85)
plt.suptitle("Topology Accuracy Comparison: FastME Options vs Standard NJ / ML Pipelines (D = 0.1 to 6.0)", 
             fontweight='bold', y=0.98)
plt.show()

# 3. 改善効果の差分ヒートマップ (FastME SPR / LG+G vs 既存手法)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4.8))

# 3-1. PSA+FastME_SPR vs PSA+NJ (SPR探索による改善: 負ならFastME SPRが良い)
p_spr = df_comp_fastme[df_comp_fastme['pipeline'] == 'PSA+FastME_SPR'].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
p_psa_nj = df_comp_fastme[df_comp_fastme['pipeline'] == 'PSA+NJ'].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
diff_spr = p_spr - p_psa_nj
sns.heatmap(diff_spr, ax=ax1, annot=True, fmt='+.3f', cmap='coolwarm', center=0, cbar_kws={'label': 'NRF Diff (FastME_SPR - PSA+NJ)'})
ax1.set_title("PSA+FastME_SPR vs PSA+NJ\n(<0: FastME_SPR Better)", fontweight='bold')
ax1.set_xlabel("Sequence Length (L)")
ax1.set_ylabel("Evolutionary Distance (D)")
ax1.invert_yaxis()

# 3-2. MSA+FastME_LG_G vs MSA+NJ (Gamma距離付きFastME vs RapidNJ: 負ならFastME LG+Gが良い)
p_msa_fastme = df_comp_fastme[df_comp_fastme['pipeline'] == 'MSA+FastME_LG_G'].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
p_msa_nj = df_comp_fastme[df_comp_fastme['pipeline'] == 'MSA+NJ'].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
diff_msa = p_msa_fastme - p_msa_nj
sns.heatmap(diff_msa, ax=ax2, annot=True, fmt='+.3f', cmap='coolwarm', center=0, cbar_kws={'label': 'NRF Diff (FastME_LG_G - MSA+NJ)'})
ax2.set_title("MSA+FastME_LG_G vs MSA+NJ\n(<0: FastME_LG_G Better)", fontweight='bold')
ax2.set_xlabel("Sequence Length (L)")
ax2.set_ylabel("Evolutionary Distance (D)")
ax2.invert_yaxis()

# 3-3. MSA+FastME_LG_G vs MSA+ML (最尤法との残余ギャップ: 正ならMLが良い)
p_ml = df_comp_fastme[df_comp_fastme['pipeline'] == 'MSA+ML'].pivot_table(index='distance', columns='length', values='nrf_distance', aggfunc='mean')
diff_ml = p_msa_fastme - p_ml
sns.heatmap(diff_ml, ax=ax3, annot=True, fmt='+.3f', cmap='coolwarm', center=0, cbar_kws={'label': 'NRF Gap (FastME_LG_G - MSA+ML)'})
ax3.set_title("MSA+FastME_LG_G vs MSA+ML\n(>0: MSA+ML Better)", fontweight='bold')
ax3.set_xlabel("Sequence Length (L)")
ax3.set_ylabel("Evolutionary Distance (D)")
ax3.invert_yaxis()

plt.suptitle("FastME Optimization Impact: Comparative Performance Gaps (D = 0.1 to 6.0)", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. 実験横断の総合比較 & 勝率サマリー

全実験条件（全 354 条件）における PSA+NJ vs MSA+ML の勝敗集計と平均 NRF distance の一覧。

In [ ]:
# 全実験条件における PSA+NJ vs MSA+ML の勝敗集計
comp_records = []
for exp_id in df_all['experiment_id'].unique():
    sub_exp = df_all[df_all['experiment_id'] == exp_id]
    if not ('PSA+NJ' in sub_exp['pipeline'].values and 'MSA+ML' in sub_exp['pipeline'].values):
        continue
    
    group_cols = ['distance', 'length']
    for extra in ['num_taxa', 'alpha', 'ics_prop']:
        if len(sub_exp[extra].unique()) > 1:
            group_cols.append(extra)
            
    psa_grp = sub_exp[sub_exp['pipeline'] == 'PSA+NJ'].groupby(group_cols)['nrf_distance'].mean()
    ml_grp = sub_exp[sub_exp['pipeline'] == 'MSA+ML'].groupby(group_cols)['nrf_distance'].mean()
    
    merged = pd.concat([psa_grp.rename('psa'), ml_grp.rename('ml')], axis=1).dropna()
    psa_wins = (merged['psa'] < merged['ml'] - 0.005).sum()
    ml_wins = (merged['ml'] < merged['psa'] - 0.005).sum()
    ties = len(merged) - psa_wins - ml_wins
    
    comp_records.append({
        'Experiment ID': exp_id,
        'Experiment Name': sub_exp['experiment_name'].iloc[0],
        'Total Conditions': len(merged),
        'PSA+NJ Wins': psa_wins,
        'MSA+ML Wins': ml_wins,
        'Ties': ties,
        'PSA+NJ Win Rate (%)': round(psa_wins / len(merged) * 100, 1),
        'Avg NRF (PSA+NJ)': round(merged['psa'].mean(), 4),
        'Avg NRF (MSA+ML)': round(merged['ml'].mean(), 4),
    })

df_winrate = pd.DataFrame(comp_records)
df_winrate

---
## 6. 配列類似度・同一性解析（Sequence Similarity & Identity Analysis）

Exp1 の全条件（$D \in [0.1..3.0]$, $L \in [100..1500]$）および高進化距離領域（$D \in [4.0, 5.0, 6.0]$, $L \in [300..1500]$）における、**各条件 100 レプリケート（計 3,700 データセット）** の配列間類似度（Sequence Identity %）を解析します。
- **True MSA Identity**: 真のアライメント（Ground Truth）に基づく真の残基一致度
- **MAFFT MSA Identity**: MAFFT 多重アライメントに基づく残基一致度
- **PSA Identity**: Needleman-Wunsch ペアワイズアライメントに基づく残基一致度

In [ ]:
# 配列類似度データのロード
sim_csv_path = SIMILARITY_DIR / "similarity_summary.csv"

if sim_csv_path.exists():
    df_sim = pd.read_csv(sim_csv_path)
    print(f"✓ Loaded {len(df_sim):,} similarity records across {len(df_sim.groupby(['distance', 'length']))} conditions.")
    
    # 1. 類似度減衰スペクトラム (D = 0.1 to 6.0)
    lengths = [300, 500, 1000, 1500]
    fig, axes = plt.subplots(1, len(lengths), figsize=(16, 4.2), sharey=True)

    for idx, length in enumerate(lengths):
        ax = axes[idx]
        sub = df_sim[df_sim['length'] == length]

        sns.lineplot(data=sub, x='distance', y='true_identity_mean', ax=ax,
                     color='#9467bd', marker='o', label='True MSA Identity', errorbar=('ci', 95), linewidth=1.8)
        sns.lineplot(data=sub, x='distance', y='mafft_identity_mean', ax=ax,
                     color='#ff7f0e', marker='s', label='MAFFT MSA Identity', errorbar=('ci', 95), linewidth=1.8)
        sns.lineplot(data=sub, x='distance', y='psa_identity_mean', ax=ax,
                     color='#1f77b4', marker='^', label='PSA Identity (NW)', errorbar=('ci', 95), linewidth=1.8)

        ax.axhline(0.05, color='gray', linestyle=':', linewidth=1.2, label='Random Expectation (5%)')

        ax.set_title(f"Length L = {length} aa", fontweight='bold')
        ax.set_xlabel("Evolutionary Distance (D)")
        ax.set_ylabel("Pairwise Sequence Identity" if idx == 0 else "")
        ax.set_ylim(-0.02, 1.02)

        if ax.get_legend():
            ax.get_legend().remove()

    handles, labels = axes[0].get_legend_handles_labels()
    axes[-1].legend(handles, labels, bbox_to_anchor=(1.05, 1.0), loc='upper left', frameon=True, title="Identity Metric")

    plt.subplots_adjust(wspace=0.08, top=0.85, right=0.82)
    plt.suptitle("Sequence Similarity Decay Spectrum across Evolutionary Distance (D = 0.1 to 6.0)", 
                 fontweight='bold', y=0.98)
    plt.show()

    # 2. アライメントバイアス（見かけ上の一致度過大評価）
    df_sim['mafft_overalignment'] = df_sim['mafft_identity_mean'] - df_sim['true_identity_mean']
    df_sim['psa_overalignment'] = df_sim['psa_identity_mean'] - df_sim['true_identity_mean']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    pivot_mafft = df_sim.pivot_table(index='distance', columns='length', values='mafft_overalignment', aggfunc='mean')
    sns.heatmap(pivot_mafft, ax=ax1, annot=True, fmt='+.3f', cmap='coolwarm', center=0,
                cbar_kws={'label': 'Mean Identity Bias (MAFFT - True)'})
    ax1.set_title("MAFFT MSA Over-alignment Bias\n(MAFFT Identity - True Identity)", fontweight='bold')
    ax1.set_xlabel("Sequence Length (L)")
    ax1.set_ylabel("Evolutionary Distance (D)")
    ax1.invert_yaxis()

    pivot_psa = df_sim.pivot_table(index='distance', columns='length', values='psa_overalignment', aggfunc='mean')
    sns.heatmap(pivot_psa, ax=ax2, annot=True, fmt='+.3f', cmap='coolwarm', center=0,
                cbar_kws={'label': 'Mean Identity Bias (PSA - True)'})
    ax2.set_title("Needleman-Wunsch PSA Over-alignment Bias\n(PSA Identity - True Identity)", fontweight='bold')
    ax2.set_xlabel("Sequence Length (L)")
    ax2.set_ylabel("Evolutionary Distance (D)")
    ax2.invert_yaxis()

    plt.suptitle("Over-alignment Bias in High Evolutionary Distance Regime (D = 0.1 to 6.0)", fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ Similarity summary CSV not found at {sim_csv_path}")

---
## 7. 全実験・全条件の Example シミュレーション実行 & 階層的保存・可視化

`analysis/analysis.md` の方針に基づき、**全実験（実験1〜10）のすべての条件** について 1 replicate ずつシミュレーションを実行し、生成された MSA・系統樹・アライメント情報を `analysis/example/<experiment_id>/<condition_name>/` 配下に実験別に階層的に保存・可視化します。

In [ ]:
# bin/generate_all_examples.py をインポートして全条件の example をロード/生成
sys.path.insert(0, str(PROJECT_ROOT / "bin"))
import generate_all_examples

example_tasks = generate_all_examples.build_all_tasks(EXAMPLE_DIR, PROJECT_ROOT)
print(f"Total Example Conditions Defined: {len(example_tasks)} conditions across experiment categories.")

max_workers = min(8, os.cpu_count() or 4)
example_records = []

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(generate_all_examples.generate_single_example, t) for t in example_tasks]
    for f in as_completed(futures):
        example_records.append(f.result())

df_example_summary = pd.DataFrame(example_records)
print(f"\n✅ All {len(df_example_summary)} conditions verified and loaded successfully!")

summary_by_exp = df_example_summary.groupby("experiment_id").agg(
    conditions_count=("condition_name", "count"),
    mean_aligned_len=("msa_aligned_length", "mean"),
    mean_psa_nrf=("pwa_nrf", "mean"),
    mean_msa_nj_nrf=("msa_nj_nrf", "mean")
).reset_index()

summary_by_exp

In [ ]:
# 任意の実験・条件を指定して可視化する汎用関数
def visualize_example(experiment_id="exp1_default", condition_name="D1.0_L500", example_base_dir=EXAMPLE_DIR):
    """
    指定した実験・条件の True Tree vs Estimated Tree のトポロジー描画、および MSA 配列長分布の可視化
    """
    cond_dir = example_base_dir / experiment_id / condition_name
    true_tree_path = cond_dir / "true_tree.nwk"
    psa_tree_path = cond_dir / "pwa_nj_tree.nwk"
    msa_nj_path = cond_dir / "msa_nj_tree.nwk"
    fasta_path = cond_dir / "unaligned.fasta"
    mafft_path = cond_dir / "mafft_msa.fasta"
    meta_path = cond_dir / "summary.json"
    
    if not (true_tree_path.exists() and psa_tree_path.exists()):
        print(f"Condition not found: {experiment_id}/{condition_name}")
        return
        
    true_tree = Phylo.read(str(true_tree_path), "newick")
    psa_tree = Phylo.read(str(psa_tree_path), "newick")
    msa_nj_tree = Phylo.read(str(msa_nj_path), "newick") if msa_nj_path.exists() else None
    
    meta = {}
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)
            
    psa_nrf = meta.get("pwa_nrf", "N/A")
    msa_nrf = meta.get("msa_nj_nrf", "N/A")
    
    # 1. 系統樹トポロジーの描画
    fig = plt.figure(figsize=(15, 6))
    ax1 = fig.add_subplot(1, 3, 1)
    ax2 = fig.add_subplot(1, 3, 2)
    ax3 = fig.add_subplot(1, 3, 3)
    
    Phylo.draw(true_tree, do_show=False, axes=ax1, branch_labels=lambda c: "")
    ax1.set_title(f"True Tree\n(Condition: {condition_name})", fontweight='bold', fontsize=11)
    
    Phylo.draw(psa_tree, do_show=False, axes=ax2, branch_labels=lambda c: "")
    ax2.set_title(f"PSA+NJ (RapidNJ)\n(nRF = {psa_nrf})", fontweight='bold', fontsize=11, color='#1f77b4')
    
    if msa_nj_tree:
        Phylo.draw(msa_nj_tree, do_show=False, axes=ax3, branch_labels=lambda c: "")
        ax3.set_title(f"MSA+NJ (MAFFT+RapidNJ)\n(nRF = {msa_nrf})", fontweight='bold', fontsize=11, color='#ff7f0e')
    else:
        ax3.axis('off')
        
    plt.suptitle(f"Topology Comparison: [{experiment_id} / {condition_name}]", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # 2. 配列長・アライメント構造の可視化
    unaligned_lens = [len(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")]
    mafft_seqs = list(SeqIO.parse(mafft_path, "fasta"))
    mafft_len = len(mafft_seqs[0].seq)
    gap_props = [rec.seq.count("-") / mafft_len for rec in mafft_seqs]
    
    fig, (ax_l, ax_g) = plt.subplots(1, 2, figsize=(12, 3.8))
    sns.histplot(unaligned_lens, bins=15, kde=True, ax=ax_l, color='skyblue')
    ax_l.set_title(f"Unaligned Sequence Length Distribution\n(Mean = {np.mean(unaligned_lens):.1f} aa, Target L = {meta.get('length', 'N/A')})", fontweight='bold')
    ax_l.set_xlabel("Sequence Length (aa)")
    
    sns.histplot(gap_props, bins=15, kde=True, ax=ax_g, color='salmon')
    ax_g.set_title(f"Gap Proportion in MAFFT MSA\n(Total Aligned Length = {mafft_len} aa)", fontweight='bold')
    ax_g.set_xlabel("Gap Proportion per Taxon")
    
    plt.tight_layout()
    plt.show()

# 代表的な各実験条件の可視化例
print("=== [Exp1: Default D=1.0, L=500] ===")
visualize_example("exp1_default", "D1.0_L500")

print("=== [Exp2: Taxon N=128, D=1.0, L=500] ===")
visualize_example("exp2_taxon", "N128_D1.0_L500")

print("=== [Exp6: ICS 20% Invariant Sites, D=1.0, L=500] ===")
visualize_example("exp6_ics", "ics0.2_D1.0_L500")

print("=== [Exp8: High Distance D=5.0, L=500] ===")
visualize_example("exp8_high_dist", "D5.0_L500")

---
## 8. まとめ & 考察

本ベンチマーク解析（全 10 実験、計 118,000 レプリケート、354 条件）および配列類似度解析（3,700 データセット）から得られた主要な結論は以下の通りです。

### 1. MSA+ML（最尤法）の全領域における圧倒的優位性
- **実証的結論**: 「短配列・高進化距離領域では多重アライメント崩壊を避ける PSA+NJ が有利になる」という仮説に反し、**実測データでは全 354 条件中 352 条件（勝率 99.4%）において `MSA+ML` が `PSA+NJ` を圧倒**しました。
- `MSA+ML` の平均 NRF distance は **0.145〜0.184** であるのに対し、`PSA+NJ` は **0.261〜0.294** と、トポロジー推定精度に顕著な開きが存在します。
- 配列長 $L$ が 100 aa と短い場合や、進化距離 $D$ が 3.0〜6.0 と極めて高い場合、Taxon 数が 128 に達する場合、ICS（短縮配列）が 20% 混入する場合でも、`MSA+ML` の優位性は揺るぎませんでした。

### 2. FastME 最適化手法（SPR 近傍探索 / ガンマ距離モデル）の性能
- **`PSA+FastME_SPR`**: 通常の RapidNJ（`PSA+NJ`）と比較して、SPR（Subtree Pruning and Regrafting）による局所探索を行うことで、高進化距離域を中心にトポロジー推定精度が向上します。
- **`MSA+FastME_LG_G`**: サイト間速度不均一性を考慮した LG+Gamma 距離モデルを用いることで、`MSA+NJ` に対し全距離域で精度が改善しますが、完全な最尤法（`MSA+ML`）との間には依然として有意な精度ギャップが存在します。

### 3. 進化距離と配列同一性・アライメントバイアス（Over-alignment）
- 進化距離 $D=0.1$ での真の一致度は約 **92.2%** ですが、$D=3.0$ で **36.5%**、$D=6.0$ では **24.7%** まで低下します。
- $D \ge 3.0$ の高距離領域では、MAFFT や Needleman-Wunsch PSA はスコア最大化のため偶然の一致を残基ペアとして強制整列させるため、**真の一致度よりも 10〜20% 高く見せかける「オーバーアライメント・バイアス」** が発生します。
- それにもかかわらず、最尤法（`MSA+ML`）は ModelFinder による速度不均一性（+G4）パラメータ推定や尤度最適化の統計的頑健性により、高変異領域でも最も高いトポロジー精度を維持しています。

### 4. PSA+NJ の位置づけと実用的意義
- **精度面**: トポロジー推定精度を最優先する用途において、`PSA+NJ` を単体で第一選択とする妥当性は薄いと言えます。
- **計算効率・用途**: 一方で、`PSA+NJ` は多重アライメントの計算コストを完全に回避できるため、Taxon 数 $N$ が数千〜数万に及ぶ超大規模データセットでの初期系統樹構築（Guide Tree 作成）や、高速なフィルタリング・クラスタリング用途において実用的な意義を持ちます。

---
## 9. 深層考察と今後の研究展望

---

### 9.1 重要な発見事項の整理

全 10 実験（計 118,000 レプリケート）を通じて、以下のパターンが一貫して観察された。

#### 9.1.1 MSA+ML の圧倒的優位性と「アライメント誤差耐性」の機序

最尤法（`MSA+ML`）は進化距離 $D=0.1 \\sim 6.0$ の全域・配列長 $L=100 \\sim 1500$ の全域にわたって最高精度を維持した。  
この結果は当初の仮説（「高変異・短配列領域では MAFFT アライメントが崩壊するため PSA が有利になるはず」）と正反対であり、以下の機序が推定される：

**仮説A（統計的尤度最適化による誤差吸収）**: IQ-TREE は「観測されたアライメント（MAFFT 出力）」に対して尤度関数を最大化する。高変異領域でアライメントが局所的に誤っていても、尤度面の形状（分岐比・枝長・置換パラメータの同時最適化）がトポロジー推定を安定させる可能性がある。

**仮説B（$\\hat{\\alpha}$ パラメータの適応的推定）**: ModelFinder により推定されるガンマ形状母数 $\\hat{\\alpha}$ は、進化距離 $D$ の増加に伴い $1.0 \\to 1.9$ と系統的に上昇する（$r = 0.72$）。高変異領域では多くのサイトが飽和しており、速度不均一性が高く見える一方で、モデルが実態に合わせて柔軟に適応していることを意味する。

**仮説C（ギャップパターン情報の利用）**: MAFFT MSA に含まれるギャップパターンは系統的信号として機能する（挿入・欠失イベントはトポロジー情報を含む）。PSA はペアワイズ比較に留まりこの情報を利用できない。

> [!NOTE]
> 仮説AとCを直接検証するには、ギャップ列を除いた配列（gap-stripped MSA）や意図的に破壊したシャッフルアライメントを入力した場合の精度を測定する「アライメント感度実験」（実験11として提案）が有効である。

---

#### 9.1.2 距離法間の精度階層と情報損失

実験結果から、以下の精度階層が成立する（NRF Distance の低い順に優れる）：

$$\\text{TRUE\\_DIST+NJ} \\ll \\text{MSA+ML} \\approx \\text{TRUE\\_MSA+ML} < \\text{MSA+FastME\\_LG\\_G} < \\text{MSA+NJ} \\approx \\text{PSA+FastME\\_SPR} < \\text{PSA+NJ}$$

`TRUE_DIST+NJ`（真の進化距離行列から NJ を適用）は **NRF = 0.0 を常に達成** する一方で、`TRUE_MSA+NJ`（真のアライメントから Poisson 距離で NJ）は `PSA+NJ` と比較して大幅に優れる。この差分は「アライメント誤差」ではなく「Poisson 距離モデルの推定誤差（飽和補正の欠如）」に起因する情報損失を意味する。

> **解釈**: 距離法の精度ボトルネックは、アライメント品質よりも「使用する距離モデルの適切さ」と「NJ アルゴリズム固有のトポロジー探索能力の限界」にある可能性が高い。

---

#### 9.1.3 ICS 条件における MSA+ML の「逆説的改善」

実験6の ICS 解析では、不変サイト割合 $\\mathrm{ics\\_prop}$ が増加するほど `MSA+ML` の NRF distance が **微減**（$0.169 \\to 0.156$）する逆説的結果が得られた。一方で `PSA+NJ` と `MSA+NJ` の精度はほぼ不変であった。

この現象の解釈：ICS によりサイトの多様性が低下して Poisson 距離の分散が縮小すると、ModelFinder が LG+I 等の不変サイトを含むモデルを正確に選択できるようになり、より適切なパラメータ推定が可能になるためと考えられる。

---

#### 9.1.4 Taxon 数スケーリングにおける各手法の脆弱性

実験2（$N = 8 \\to 128$、$D=1.0$、$L=500$）では Taxon 数増加に伴い全手法の NRF distance が増加するが、その増加率に差がある：

| Taxon 数 $N$ | PSA+NJ | MSA+NJ | MSA+ML |
|:---:|:---:|:---:|:---:|
| 8   | 0.140 | 0.142 | 0.087 |
| 16  | 0.184 | 0.192 | 0.102 |
| 64  | 0.254 | 0.269 | 0.129 |
| 128 | 0.298 | 0.302 | 0.130 |

`MSA+ML` の精度劣化は $N=64 \\to 128$ でほぼ飽和するのに対し、`PSA+NJ` と `MSA+NJ` は線形に近い劣化を示す。これは最尤法が Taxon 数増加に伴う探索空間の拡大を局所探索（NNI/SPR）で補完できることを示唆する。しかし **$N \\ge 256$ の領域は未検証**であり、実用規模での挙動解明が課題として残る。

---

### 9.2 今後の実験提案

以下では、現在の結果から生じた未解決問題を解明するための追加実験を優先度順に提案する。

---

#### 💡 実験11（優先度：高）: アライメント感度実験（Alignment Sensitivity Analysis）

**目的**: MSA+ML の「アライメント誤差への頑健性」の機序解明  
**背景**: なぜ MAFFT アライメントが 92% → 25% まで品質劣化する高距離域でも MSA+ML が精度を維持できるのか？尤度最適化か、ギャップ情報か、アミノ酸組成か —— 何が精度の鍵を握るかを切り分ける。

**アライメント入力の比較設計**:
| 入力アライメント | 説明 |
|---|---|
| `TRUE_MSA` | シミュレーション上の真のアライメント（理論上限） |
| `MAFFT_MSA` | MAFFT によるアライメント（現在の実験） |
| `MAFFT_GAPSTRIP` | ギャップ列を除いた MSA（ギャップ情報なし） |
| `SHUFFLED_MSA` | 各タキソンのアミノ酸はそのままで、サイト順をランダムシャッフル（サイト間独立性を破壊） |
| `RANDOM_MSA` | 完全ランダムなアミノ酸配列をアライメントとして入力（ゼロ系統情報） |

**期待される結果**: `SHUFFLED_MSA` でも `PSA+NJ` を上回れば MSA+ML の優位性は「アライメントの正確さ」に依存しないことが示される。また `GAPSTRIP` と `MAFFT_MSA` の精度差が小さければ、ギャップ列は系統情報よりノイズとして機能していることを意味する。

**パラメータ**: $D \\in [1.0, 3.0, 6.0]$, $L \\in [500, 1000]$, $N=32$, Replicates=100

---

#### 💡 実験12（優先度：高）: 置換モデルのミスマッチ実験（Model Misspecification）

**目的**: 「シミュレーションに使ったモデルと異なるモデルが選択される」状況における MSA+ML の頑健性定量化  
**背景**: 高変異域（$D=3.0$）では ModelFinder が推定する $\\hat{\\alpha} \\approx 1.9$ が真値 $\\alpha=1.0$ から大きく乖離している。この系統的モデルミスマッチがトポロジー精度に与えるインパクトを明示的に検証する。

**設計案**:
1. `WAG+G4` でシミュレーション → `LG+G4` 固定モデルで MSA+ML（意図的ミスマッチ）vs ModelFinder 自動選択
2. `JTT+G4` 生成 → `LG+G4`、`WAG+G4`、ModelFinder の 3 条件を比較
3. 真モデル一致下（`LG+G4` 生成 → `LG+G4` 固定指定）vs ModelFinder 自動選択の精度差を全距離域で測定

**パラメータ**: $D \\in [0.1, 1.0, 3.0, 6.0]$, $L \\in [300, 1000]$, $N=32$, Replicates=100

---

#### 💡 実験13（優先度：中）: 超大規模 Taxon 数スケーリング（$N \\ge 256$）と計算コスト評価

**目的**: 実用的なゲノミクス規模（$N \\ge 256$）での精度 vs 計算コストのトレードオフ解明  
**背景**: ゲノム解析では $N \\ge 1000$ が一般的だが、現在の測定は $N=128$ で終了している。$N \\ge 256$ では MAFFT の多重アライメントが計算ボトルネックとなり、FastME や PSA 系手法が計算効率面で逆転する可能性がある。

**設計**:
- Taxon 数: $N \\in [256, 512, 1024]$（対数スケール）
- 進化距離: $D \\in [1.0, 3.0]$
- 配列長: $L \\in [500, 1000]$
- 評価指標: NRF distance に加え、**CPU time / Memory usage を必須計測**
- 対象手法: `PSA+FastME_SPR`, `MSA+NJ` (MAFFT+RapidNJ), `MSA+ML`（IQ-TREE, threads=4）

**期待される洞察**: 精度と計算コストのトレードオフ交差点を特定する。例えば $N > 512$ で `MSA+ML` の計算時間が `PSA+FastME_SPR` の 100 倍以上になる場合、高速手法の実用的優位性が生まれる。

---

#### 💡 実験14（優先度：中）: 不均一速度・非定常進化モデルでのシミュレーション

**目的**: 現在の実験設定（時間均一 LG+G4）が最も有利に機能する可能性を排除し、より現実的な進化モデルでの頑健性を検証

**設計案**:
1. **時間不均一進化（Non-stationary evolution）**: 系統によって異なる平衡頻度を持つ CAT モデル（PHYLOBAYES）でシミュレーション
2. **Covarion モデル**: 進化速度がサイトごとかつ時間的に変動するモデル（一部のサイトが「on/off」を切り替える）
3. **分岐長の不均一性（星型放散・長枝収引）**: ランダム Yule 樹の代わりに、長い内部枝や短い放射状樹を意図的に生成し、Long Branch Attraction（LBA）アーティファクトへの各手法の耐性を評価

---

#### 💡 実験15（優先度：中）: PSA 代替距離推定手法の探索

**目的**: NW（Needleman-Wunsch）ペアワイズアライメント以外の高速・高精度距離推定手法との比較  
**背景**: 現在の `PSA+NJ` の精度ボトルネックは、高変異域（$D \\ge 3.0$, True Identity $\\le 36\\%$）での Poisson 距離の飽和補正の欠如にある。以下の代替手法が精度改善に寄与する可能性がある：

| 手法カテゴリー | 具体的手法 | 特徴 |
|---|---|---|
| プロファイルアライメント | HH-suite (HHpred) | ドメインプロファイル（HMM-HMM）を用いた高精度遠距離アライメント |
| k-mer ベース距離 | Mash / AAF (alignment-free) | アライメント不要・超高速・大規模スケーラブル |
| ペアワイズ ML 距離 | PAML `codeml` / FastTree 枝長推定 | より精密な進化距離（尤度ベース）による NJ 入力行列の構築 |
| プロファイル付き NJ | FastTree (approximate ML) | 完全な最尤法よりも高速で NJ より高精度なトポロジー探索 |

---

#### 💡 実験16（優先度：低）: 実データ検証（Real Biological Sequences Benchmark）

**目的**: シミュレーションで明らかになった精度傾向が実際の生物配列でも再現されるかの検証  
**背景**: AliSim による LG+G4 シミュレーションでは成立しない現実的な進化の複雑さ（水平遺伝子転送・収束進化・組み換え・組成バイアス）が実データには含まれる。

**設計**:
- データソース: Pfam A ファミリーから既知の分類学的系統関係を持つタンパク質ファミリーを選定
- 参照系統樹: NCBI Taxonomy による分類学的系統樹
- 各手法の精度を実データで評価し、シミュレーション結果との差異（乖離スコア）を定量化

---

### 9.3 方法論上の注意事項と限界

1. **シミュレーターの仮定**: AliSim は時間均一な LG+G4 モデルを仮定しているが、実際の進化は位置依存的な速度変動・組成バイアスを持つ。実データへの一般化可能性には留意が必要。

2. **距離指標の限界**: 本研究では Robinson-Foulds 距離（nRF）のみをトポロジー精度の指標として使用している。nRF は「分岐の一致」をカウントするだけで、「どの分岐で誤っているか」「主要な単系統群の誤り率」等の情報を持たない。今後は **Quartet Distance** や **Weighted RF Distance** による補完的評価を導入すべきである。

3. **樹形の偏り**: 全実験で樹形はランダム（Yule モデル）を使用しているが、実際の生物進化では星型放散（star-tree）や長い内部枝（long internal branch）等の偏ったトポロジーが多く観察される。Long Branch Attraction（LBA）アーティファクトが距離法を誤りに誘導することが知られており、これらの条件下での各手法の比較は未解決課題である。

4. **$\\hat{\\alpha}$ の系統的過大推定**: 実験1で ModelFinder が高変異域で $\\hat{\\alpha} \\approx 1.9$ を推定するのは、真値 $\\alpha=1.0$ からの系統的乖離を意味する。この推定誤りが最終的なトポロジー精度に与える影響の定量的評価（実験12の対象）は未解決の重要課題である。

5. **ギャップモデルの欠如**: 本研究の PSA および MSA いずれも、ギャップ（インデル）を系統推定に陽に組み込んでいない。インデルを明示的にモデル化する手法（PhyloBayes Indel, IQTREE `-fconst`, StatAlign 等）との比較も、アライメント崩壊領域での精度改善に寄与する可能性がある。